## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import os
DRIVE_PROJECT = "/content/drive/MyDrive/malaria_densenet"
os.makedirs(DRIVE_PROJECT, exist_ok=True)
os.makedirs(f"{DRIVE_PROJECT}/saved_models", exist_ok=True)
os.makedirs(f"{DRIVE_PROJECT}/logs/fit", exist_ok=True)
print("Project folder ready:", DRIVE_PROJECT)

In [ ]:
import tensorflow as tf
print("TF:", tf.__version__, " GPU:", tf.test.gpu_device_name())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                              confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc)

## 2. Data Splitting

In [ ]:
cell_images_dir = "cell_images"

if not os.path.exists(cell_images_dir):
    !rm -rf cell_images.zip cell_images
    !wget -q https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip
    !unzip -q cell_images.zip
else:
    print("cell_images already present locally, skipping download.")

In [ ]:
SEED = 42
IMG = 224
BATCH = 32
TRAIN_FRAC = 0.8
VAL_FRAC = 0.1

tf.keras.utils.set_random_seed(SEED)

File paths and labels are collected, sorted, and shuffled once with a seeded numpy permutation, then cut 80/10/10.

In [ ]:
class_names = sorted(os.listdir(cell_images_dir))
INFECTED = class_names.index("Parasitized")
print("Classes:", class_names, " infected index:", INFECTED)

VALID_EXTS = (".png", ".jpg", ".jpeg")

all_paths, all_labels = [], []
for label_idx, cls in enumerate(class_names):
    cls_dir = os.path.join(cell_images_dir, cls)
    for fname in sorted(os.listdir(cls_dir)):
        if fname.lower().endswith(VALID_EXTS):
            all_paths.append(os.path.join(cls_dir, fname))
            all_labels.append(1.0 if label_idx == INFECTED else 0.0)

all_paths = np.array(all_paths)
all_labels = np.array(all_labels, dtype=np.float32)
n = len(all_paths)
print(f"Total valid images: {n}")

rng = np.random.RandomState(SEED)
perm = rng.permutation(n)
all_paths, all_labels = all_paths[perm], all_labels[perm]

n_train = int(TRAIN_FRAC * n)
n_val = int(VAL_FRAC * n)

train_paths, train_labels = all_paths[:n_train], all_labels[:n_train]
val_paths, val_labels = all_paths[n_train:n_train+n_val], all_labels[n_train:n_train+n_val]
test_paths, test_labels = all_paths[n_train+n_val:], all_labels[n_train+n_val:]

print(f"total={n}  train={len(train_paths)}  val={len(val_paths)}  test={len(test_paths)}")

## 3. Pipeline Preprocessing

In [ ]:
def load_and_prep(path, label):
  """Read one image file and prepare it for the network.

    Args:
        path: scalar string tensor with the image file path.
        label: float32 scalar, 1.0 for infected and 0.0 for uninfected.

    Returns:
        (image, label): image is a float32 tensor of shape (IMG, IMG, 3) with
        raw 0-255 pixel values. Normalisation is deliberately left to the
        model's own preprocess_input layer so each backbone applies its own.
    """
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG, IMG))
    return img, label

def make_ds_from_paths(paths, labels, training=False):
  """Build a batched tf.data pipeline from fixed arrays of paths and labels.

    The split itself is fixed beforehand in numpy, so this function cannot
    move an image between train, validation and test.

    Args:
        paths, labels: numpy arrays for one split.
        training: if True, shuffle within the split (order only) each epoch.

    Returns:
        A prefetched dataset yielding (images, labels) batches of size BATCH.
    """
    d = tf.data.Dataset.from_tensor_slices((paths, labels))
    d = d.map(load_and_prep, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        d = d.shuffle(2048, seed=SEED)
    return d.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds_from_paths(train_paths, train_labels, training=True)
val_ds = make_ds_from_paths(val_paths, val_labels)
test_ds = make_ds_from_paths(test_paths, test_labels)

In [ ]:
y_check_1 = np.concatenate([y.numpy() for _, y in test_ds])
y_check_2 = np.concatenate([y.numpy() for _, y in test_ds])
print("Stable across iterations:", np.array_equal(y_check_1, y_check_2))
print("Test set size:", len(y_check_1), " infected:", int(y_check_1.sum()))

In [ ]:
for name, labels_arr in [("train", train_labels), ("val", val_labels), ("test", test_labels)]:
    print(f"{name:5s} n={len(labels_arr):6d}  infected={labels_arr.mean():.1%}")

In [ ]:
imgs, ys = next(iter(train_ds))
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, im, y in zip(axes.ravel(), imgs[:8], ys[:8]):
    ax.imshow(im.numpy().astype("uint8"))
    ax.set_title("INFECTED" if y == 1 else "uninfected")
    ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Model

In [ ]:
augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
], name="augmentation")

In [ ]:
def set_backbone_trainable(base, unfreeze_prefixes):
  """Freeze the backbone except layers whose names start with given prefixes.

    BatchNormalization layers always stay frozen, even inside unfrozen blocks,
    to keep their pretrained running statistics.

    Args:
        base: the DenseNet121 backbone model.
        unfreeze_prefixes: list such as ["conv5"] or ["conv4", "conv5"];
            an empty list leaves the whole backbone frozen.
    """
    base.trainable = True
    for layer in base.layers:
        layer.trainable = any(layer.name.startswith(p) for p in unfreeze_prefixes)
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

Design rationale: model

DenseNet121 with ImageNet weights is used as a feature extractor with a new head (global average pooling, optional dropout, one sigmoid unit). Its parameter-efficient dense blocks (Huang et al., 2017) suit a dataset of about 22k training images. Early layers are assumed to hold general features and later layers task-specific ones, so only the last dense block (conv5) is ever unfrozen. BatchNorm layers stay frozen and the backbone is called with training=False so pretrained statistics are kept. Preprocessing sits inside the model so each backbone can use its own.

In [ ]:
def build_model(cfg):
  """Assemble the DenseNet121 transfer-learning classifier.

    Pipeline: input -> optional augmentation -> DenseNet preprocess_input ->
    DenseNet121 (ImageNet weights, include_top=False) -> global average pooling
    -> optional dropout -> Dense(1, sigmoid) giving P(infected).

    Args:
        cfg: dict with keys "augment" (bool), "unfreeze_prefixes" (list),
            "dropout" (float). Missing keys fall back to off/frozen/0.0.

    Returns:
        (model, base): the full model and the backbone, which is called with
        training=False so BatchNorm always uses pretrained statistics.
    """
    inputs = layers.Input(shape=(IMG, IMG, 3))
    x = inputs
    if cfg.get("augment", False):
        x = augmentation(x)
    x = layers.Lambda(preprocess_input)(x)

    base = DenseNet121(include_top=False, weights="imagenet", input_shape=(IMG, IMG, 3))
    set_backbone_trainable(base, cfg.get("unfreeze_prefixes", []))

    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    if cfg.get("dropout", 0.0) > 0:
        x = layers.Dropout(cfg["dropout"])(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return models.Model(inputs, outputs), base

## 5. Experiment tracking

In [ ]:
RESULTS_PATH = f"{DRIVE_PROJECT}/results.json"
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        results = json.load(f)
    print(f"Reloaded {len(results)} past experiments.")
else:
    results = []
    print("No past results found.")

One call equals one logged experiment. Validation metrics come from a single pass so labels and predictions stay aligned. Each run saves a TensorBoard log, a row in results.json, and a .keras model on Drive. Experiments 1-8 used the earlier, unstable split; exp_10 used the corrected one.

In [ ]:
experiment_counter = {"n": len(results)}

def make_run_name(label, is_repeat=False):
  """Return the next run name, e.g. densenet_exp_04_unfreeze_dropout.

    Increments experiment_counter on every call, so the number counts calls
    to this function rather than distinct configurations. Pass is_repeat=True
    to append "_repeat" for deliberate reruns.
    """
    experiment_counter["n"] += 1
    suffix = "_repeat" if is_repeat else ""
    return f"densenet_exp_{experiment_counter['n']:02d}_{label}{suffix}"

def make_tb_callback(run_name):
    """Return a Keras TensorBoard callback writing to logs/fit/<run_name> on Drive."""
    return tf.keras.callbacks.TensorBoard(log_dir=f"{DRIVE_PROJECT}/logs/fit/{run_name}", histogram_freq=1)

def load_saved_model(run_name):
  """Reload a saved .keras model from Drive.

    custom_objects supplies preprocess_input, which the Lambda layer stores by
    name only. safe_mode=False is needed to deserialize that Lambda layer;
    only use it on files you created yourself.
    """
    return tf.keras.models.load_model(
        f"{DRIVE_PROJECT}/saved_models/{run_name}.keras",
        custom_objects={"preprocess_input": preprocess_input},
        safe_mode=False,
    )

In [ ]:
def run_experiment(label, cfg, epochs=10, patience=3, is_repeat=False, notes=""):
  """Train, validate, log and save one experiment.

    Builds the model from cfg, compiles it with Adam (or SGD with momentum 0.9
    if cfg["optimizer"] == "sgd_momentum"), trains with early stopping on
    val_loss (restoring the best weights), and logs to TensorBoard. Validation
    metrics are computed in a single pass that collects labels and predictions
    together, because two separate passes over a parallel tf.data pipeline are
    not guaranteed to visit examples in the same order.

    Args:
        label: short description used in the run name.
        cfg: dict with "lr" plus optional "augment", "unfreeze_prefixes",
            "dropout", "optimizer", "batch_size".
        epochs: maximum epochs; patience: early-stopping patience.
        is_repeat, notes: bookkeeping saved with the result.

    Returns:
        (model, base, history, metrics). Also appends to results.json and saves
        the model to Drive. Metrics are on the validation split, not test.
    """
    run_name = make_run_name(label, is_repeat=is_repeat)
    print(f"\n=== Running {run_name} ===")
    print("Config:", cfg)

    model, base = build_model(cfg)
    if cfg.get("optimizer") == "sgd_momentum":
        opt = tf.keras.optimizers.SGD(learning_rate=cfg["lr"], momentum=0.9)
    else:
        opt = tf.keras.optimizers.Adam(learning_rate=cfg["lr"])
    model.compile(optimizer=opt, loss="binary_crossentropy", metrics=["accuracy"])

    exp_batch = cfg.get("batch_size", BATCH)
    if exp_batch == BATCH:
        exp_train_ds, exp_val_ds = train_ds, val_ds
    else:
        exp_train_ds = make_ds_from_paths(train_paths, train_labels, training=True)
        exp_val_ds = make_ds_from_paths(val_paths, val_labels)
        exp_train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels)) \
            .map(load_and_prep, num_parallel_calls=tf.data.AUTOTUNE).shuffle(2048, seed=SEED) \
            .batch(exp_batch).prefetch(tf.data.AUTOTUNE)
        exp_val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels)) \
            .map(load_and_prep, num_parallel_calls=tf.data.AUTOTUNE).batch(exp_batch).prefetch(tf.data.AUTOTUNE)

    tb_callback = make_tb_callback(run_name)
    early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True)

    history = model.fit(exp_train_ds, validation_data=exp_val_ds, epochs=epochs,
                         callbacks=[tb_callback, early_stop], verbose=1)

    y_true_list, y_prob_list = [], []
    for x_batch, y_batch in exp_val_ds:
        y_true_list.append(y_batch.numpy())
        y_prob_list.append(model.predict(x_batch, verbose=0).ravel())
    y_true = np.concatenate(y_true_list)
    y_prob = np.concatenate(y_prob_list)
    y_pred = (y_prob >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    metrics = {
        "accuracy": (tp + tn) / (tp + tn + fp + fn),
        "precision": precision_score(y_true, y_pred),
        "recall_sensitivity": recall_score(y_true, y_pred),
        "specificity": tn / (tn + fp),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob),
    }
    print("Validation metrics:", metrics)

    results.append({"run_name": run_name, "label": label, "is_repeat": is_repeat,
                     "notes": notes, "patience": patience, **cfg, **metrics})
    with open(RESULTS_PATH, "w") as f:
        json.dump(results, f, indent=2)

    model.save(f"{DRIVE_PROJECT}/saved_models/{run_name}.keras")
    print(f"Saved model to {run_name}.keras")
    return model, base, history, metrics

## 6. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {DRIVE_PROJECT}/logs/fit

## 7. Experiments


Each run changes one main thing from the previous best. The frozen baseline is the control. Augmentation is tested frozen, then with conv5 unfrozen, to check whether it only helps when there are trainable features to benefit. Unfreezing depth, dropout, optimizer and batch size follow. The learning rate was lowered from 1e-3 to 1e-4 at the first unfreezing run because fine-tuning pretrained weights needs smaller steps. This means that run changed two things at once, and its improvement can't be credited to unfreezing alone. Runs use one seed each, so gaps of a few tenths of a point are within noise.

In [ ]:
# Experiment 1: frozen baseline
model, base, history, metrics = run_experiment(
    "frozen_baseline",
    {"augment": False, "unfreeze_prefixes": [], "dropout": 0.0, "lr": 1e-3},
)

In [ ]:
# Experiment 2: augmentation, backbone still frozen
model, base, history, metrics = run_experiment(
    "augmentation",
    {"augment": True, "unfreeze_prefixes": [], "dropout": 0.0, "lr": 1e-3},
)

In [ ]:
# Experiment 3: unfreeze conv5 (last dense block)
model, base, history, metrics = run_experiment(
    "unfreeze_last_block",
    {"augment": False, "unfreeze_prefixes": ["conv5"], "dropout": 0.0, "lr": 1e-4},
    epochs=15,
)

In [ ]:
# Experiment 4: unfreeze conv5 + dropout
model, base, history, metrics = run_experiment(
    "unfreeze_dropout",
    {"augment": False, "unfreeze_prefixes": ["conv5"], "dropout": 0.3, "lr": 1e-4},
    epochs=15,
)

In [ ]:
# Experiment 5: unfreeze conv4 + conv5
model, base, history, metrics = run_experiment(
    "unfreeze_conv4_conv5",
    {"augment": False, "unfreeze_prefixes": ["conv4", "conv5"], "dropout": 0.0, "lr": 1e-4},
    epochs=15,
)

In [ ]:
# Experiment 6: unfreeze conv5 + augmentation
model, base, history, metrics = run_experiment(
    "unfreeze_augmentation",
    {"augment": True, "unfreeze_prefixes": ["conv5"], "dropout": 0.0, "lr": 1e-4},
    epochs=15,
)

In [ ]:
# Experiment 7: SDG Optimizer
model, base, history, metrics = run_experiment(
    "unfreeze_sgd_momentum",
    {"augment": False, "unfreeze_prefixes": ["conv5"], "dropout": 0.0, "lr": 1e-3, "optimizer": "sgd_momentum"},
    epochs=15,
)

In [ ]:
# Experiment 8: batch size
model, base, history, metrics = run_experiment(
    "unfreeze_batch64",
    {"augment": False, "unfreeze_prefixes": ["conv5"], "dropout": 0.0, "lr": 1e-4, "batch_size": 64},
    epochs=15,
)

In [ ]:
# Experiment 3 RETRAIN unfreeze conv5, under the CORRECTED stable split.
model, base, history, metrics = run_experiment(
    "unfreeze_last_block_v2",
    {"augment": False, "unfreeze_prefixes": ["conv5"], "dropout": 0.0, "lr": 1e-4},
    epochs=15,
    notes="Retrain of exp_03 under the corrected, stable train/val/test split "
          "(fixed a tf.data reshuffle-on-iteration bug in the original split).",
)

In [ ]:
pd.DataFrame(results)

Final model: held-out test evaluation

The retrained unfreeze_last_block_v2 model is evaluated once on the test split. Images, labels and probabilities are gathered in one loop so they stay aligned. The 0.5 threshold gives the confusion matrix; the ROC curve uses the raw probabilities across all thresholds.

In [ ]:
final_run_name = next(r["run_name"] for r in results if "unfreeze_last_block_v2" in r["run_name"])
print("Using:", final_run_name)

final_model = load_saved_model(final_run_name)
final_model.summary()

In [ ]:
test_images_list, y_true_list, y_prob_list = [], [], []
for x_batch, y_batch in test_ds:
    test_images_list.extend(x_batch.numpy())
    y_true_list.append(y_batch.numpy())
    y_prob_list.append(final_model.predict(x_batch, verbose=0).ravel())

test_images = np.array(test_images_list)
y_true = np.concatenate(y_true_list)
y_prob = np.concatenate(y_prob_list)
y_pred = (y_prob >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
print(f"TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"Accuracy: {(tp+tn)/len(y_true):.4f}  Sensitivity: {tp/(tp+fn):.4f}  Specificity: {tn/(tn+fp):.4f}")

In [ ]:
from tensorboard.backend.event_processing import event_accumulator

def load_tb_scalars(run_name, tag):
  """Read per-epoch train and validation values for one TensorBoard tag.

    This TF version stores epoch metrics as tensor events, so they are read
    with ea.Tensors() instead of ea.Scalars().

    Returns:
        dict {"train": (steps, values), "validation": (steps, values)}.
    """
    out = {}
    for split in ["train", "validation"]:
        log_dir = f"{DRIVE_PROJECT}/logs/fit/{run_name}/{split}"
        ea = event_accumulator.EventAccumulator(log_dir, size_guidance={"tensors": 0})
        ea.Reload()
        events = ea.Tensors(tag)
        out[split] = ([e.step for e in events], [tf.make_ndarray(e.tensor_proto).item() for e in events])
    return out

acc_data = load_tb_scalars(final_run_name, "epoch_accuracy")
loss_data = load_tb_scalars(final_run_name, "epoch_loss")

fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc_val = auc(fpr, tpr)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

axes[0].plot(*acc_data["train"], label="Train", marker="o")
axes[0].plot(*acc_data["validation"], label="Validation", marker="o")
axes[0].set_title("Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy"); axes[0].legend()

axes[1].plot(*loss_data["train"], label="Train", marker="o")
axes[1].plot(*loss_data["validation"], label="Validation", marker="o")
axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss"); axes[1].legend()

ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=["Uninfected", "Infected"],
                                         cmap="Blues", ax=axes[2])
axes[2].set_title("Confusion Matrix (Test)")

axes[3].plot(fpr, tpr, label=f"AUC = {roc_auc_val:.4f}")
axes[3].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
axes[3].set_xlabel("False Positive Rate"); axes[3].set_ylabel("True Positive Rate")
axes[3].set_title("ROC Curve (Test)"); axes[3].legend()

fig.suptitle(f"Final Model Evaluation — {final_run_name}", fontsize=13, y=1.03)
plt.tight_layout()
plt.savefig(f"{DRIVE_PROJECT}/final_all_four_visualizations.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
false_negatives = np.where((y_true == 1) & (y_pred == 0))[0]
false_positives = np.where((y_true == 0) & (y_pred == 1))[0]
print(f"False negatives: {len(false_negatives)}  False positives: {len(false_positives)}")

Most confident errors

False negatives are infected cells with the lowest predicted probability. False positives are uninfected cells with the highest. Confidently wrong cases are shown because borderline errors are less informative.

In [ ]:
fn_confidence = y_prob[false_negatives]
fp_confidence = y_prob[false_positives]

most_confident_fn = false_negatives[np.argsort(fn_confidence)[:3]]
most_confident_fp = false_positives[np.argsort(fp_confidence)[-3:]]

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for i, idx in enumerate(most_confident_fn):
    axes[0, i].imshow(test_images[idx].astype("uint8"))
    axes[0, i].set_title(f"FN — true: infected\npred prob: {y_prob[idx]:.3f}")
    axes[0, i].axis("off")

for i, idx in enumerate(most_confident_fp):
    axes[1, i].imshow(test_images[idx].astype("uint8"))
    axes[1, i].set_title(f"FP — true: uninfected\npred prob: {y_prob[idx]:.3f}")
    axes[1, i].axis("off")

fig.suptitle(f"Most confident misclassifications — {final_run_name}", fontsize=13)
plt.tight_layout()
plt.savefig(f"{DRIVE_PROJECT}/final_misclassified_examples.png", dpi=150)
plt.show()

In [ ]:
base = final_model.get_layer("densenet121")
for layer in base.layers[-6:]:
    print(layer.name, type(layer).__name__, layer.output.shape)

In [ ]:
last_conv_layer_name = "relu"

conv_extractor = tf.keras.Model(base.input, base.get_layer(last_conv_layer_name).output)

gap_layer = next(l for l in final_model.layers if isinstance(l, tf.keras.layers.GlobalAveragePooling2D))
dense_layer = next(l for l in final_model.layers if isinstance(l, tf.keras.layers.Dense))

conv_shape = conv_extractor.output_shape[1:]
classifier_input = tf.keras.Input(shape=conv_shape)
x = gap_layer(classifier_input)
x = dense_layer(x)
classifier_model = tf.keras.Model(classifier_input, x)

print("conv_extractor output shape:", conv_extractor.output_shape)
print("classifier_model summary:")
classifier_model.summary()

In [ ]:
def make_gradcam_heatmap(img_array_raw):
  """Compute a Grad-CAM heatmap for the P(infected) output.

    Gradients of the sigmoid output are taken with respect to the final
    DenseNet feature map (layer "relu", shape 7x7x1024), averaged over space
    to weight each channel, then summed over channels, ReLU'd and scaled to
    [0, 1].

    Args:
        img_array_raw: one (224, 224, 3) image with raw 0-255 values.

    Returns:
        (heatmap, probability): a 7x7 numpy array and the model's P(infected).
    """
    img_preprocessed = preprocess_input(np.expand_dims(img_array_raw.copy(), axis=0))

    with tf.GradientTape() as tape:
        conv_output = conv_extractor(img_preprocessed)
        tape.watch(conv_output)
        preds = classifier_model(conv_output)
        class_score = preds[:, 0]

    grads = tape.gradient(class_score, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_output = conv_output[0]
    heatmap = tf.reduce_sum(conv_output * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), float(preds[0, 0])

In [ ]:
correct_infected = np.where((y_true == 1) & (y_pred == 1))[0][0]
correct_uninfected = np.where((y_true == 0) & (y_pred == 0))[0][0]
incorrect_example = most_confident_fp[-1]

Grad-CAM

The model is split into a feature extractor (up to DenseNet's final relu, 7×7×1024) and a classifier head (global average pooling and Dense). Gradients of the infection probability with respect to the feature map weight each channel, and the weighted sum is shown as a heatmap. Warm colours mark regions that increased the infection score. The map is only 7×7 before upsampling, so it indicates rough regions, not exact boundaries.

In [ ]:
import matplotlib.cm as cm

def overlay_heatmap(img_raw, heatmap, alpha=0.4):
  """Upsample a 7x7 heatmap to image size, colour it with 'jet' and blend it
    over the original image. Returns a uint8 RGB image. The upsampling is
    interpolation, so the overlay looks smoother than the real resolution."""
    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], (224, 224)).numpy().squeeze()
    heatmap_uint8 = np.uint8(255 * heatmap_resized)

    jet = cm.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap_uint8]
    jet_heatmap = np.uint8(jet_heatmap * 255)

    overlay = jet_heatmap * alpha + img_raw
    overlay = np.clip(overlay, 0, 255).astype("uint8")
    return overlay

examples = [
    ("Correct — Infected", correct_infected),
    ("Correct — Uninfected", correct_uninfected),
    ("Incorrect — FP (true: uninfected)", incorrect_example),
]

fig, axes = plt.subplots(2, 3, figsize=(13, 9))

for i, (title, idx) in enumerate(examples):
    img_raw = test_images[idx]
    heatmap, pred_prob = make_gradcam_heatmap(img_raw)
    overlay = overlay_heatmap(img_raw, heatmap)

    axes[0, i].imshow(img_raw.astype("uint8"))
    axes[0, i].set_title(f"{title}\ntrue={int(y_true[idx])}  pred={y_pred[idx]}  prob={pred_prob:.3f}")
    axes[0, i].axis("off")

    axes[1, i].imshow(overlay)
    axes[1, i].set_title("Grad-CAM overlay")
    axes[1, i].axis("off")

fig.suptitle(f"Grad-CAM — {final_run_name}", fontsize=14)
plt.tight_layout()
plt.savefig(f"{DRIVE_PROJECT}/final_gradcam_examples.png", dpi=150)
plt.show()